# Phase 5 — Agentic AI Layer
**Goal:** Build dual-model autonomous decision engine. Perceive → Reason → Act → Learn.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import os, warnings
warnings.filterwarnings('ignore')

xgb_ulb  = joblib.load('../models/xgboost_model.pkl')
xgb_ieee = joblib.load('../models/xgboost_ieee_model.pkl')
scaler   = joblib.load('../data/scaler.pkl')
print("Both models loaded!")

In [ ]:
# Fraud Ledger — agent memory
LEDGER_PATH = '../data/fraud_ledger.csv'

def init_ledger():
    if not os.path.exists(LEDGER_PATH):
        pd.DataFrame(columns=['timestamp','transaction_id','amount','hour',
                               'dataset_source','ulb_fraud_prob','ieee_fraud_prob',
                               'final_fraud_prob','action','reason']
                    ).to_csv(LEDGER_PATH, index=False)
        print("Ledger created!")
    else:
        print(f"Ledger loaded — {len(pd.read_csv(LEDGER_PATH))} past decisions")

def log_to_ledger(txn_id, amount, hour, source, ulb_p, ieee_p, final_p, action, reason):
    pd.DataFrame([{
        'timestamp':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'transaction_id':txn_id, 'amount':round(float(amount),2),
        'hour':hour, 'dataset_source':source,
        'ulb_fraud_prob':round(float(ulb_p),4),
        'ieee_fraud_prob':round(float(ieee_p),4),
        'final_fraud_prob':round(float(final_p),4),
        'action':action, 'reason':reason
    }]).to_csv(LEDGER_PATH, mode='a', header=False, index=False)

init_ledger()

In [ ]:
# Action functions
def block_transaction(txn_id, amount, prob, source):
    print(f"\n🔴 BLOCK | {txn_id} | ₹{amount:.2f} | prob={prob:.4f} | {source}")
    print(f"   Card frozen. Bank alerted.")
    return "BLOCK"

def send_otp(txn_id, amount, prob, source):
    print(f"\n🟡 OTP   | {txn_id} | ₹{amount:.2f} | prob={prob:.4f} | {source}")
    print(f"   OTP sent. Transaction held.")
    return "OTP_CHALLENGE"

def allow_transaction(txn_id, amount, prob, source):
    print(f"\n🟢 ALLOW | {txn_id} | ₹{amount:.2f} | prob={prob:.4f} | {source}")
    print(f"   Approved. Pattern stored.")
    return "ALLOW"

In [ ]:
# Dual-model Agentic Decision Engine
def pad_features(features, target_length):
    f = np.array(features).flatten()
    if len(f) < target_length:
        return np.concatenate([f, np.zeros(target_length - len(f))])
    return f[:target_length]

def agentic_decision_engine(features_ulb, features_ieee, meta, source="ULB"):
    txn_id = meta.get('id','TXN_?')
    amount = meta.get('amount', 0.0)
    hour   = meta.get('hour', 0)

    print(f"\n{'─'*50}")
    print(f"  PROCESSING: {txn_id}  [{source}]")

    ulb_prob  = float(xgb_ulb.predict_proba(np.array(features_ulb).reshape(1,-1))[0][1])
    ieee_prob = float(xgb_ieee.predict_proba(np.array(features_ieee).reshape(1,-1))[0][1])
    final_prob = 0.5 * ulb_prob + 0.5 * ieee_prob

    print(f"  ULB  : {ulb_prob:.4f} | IEEE: {ieee_prob:.4f} | Final: {final_prob:.4f}")

    if final_prob > 0.8:
        action = block_transaction(txn_id, amount, final_prob, source)
        reason = f"prob {final_prob:.4f} > 0.80"
    elif final_prob > 0.5:
        action = send_otp(txn_id, amount, final_prob, source)
        reason = f"prob {final_prob:.4f} between 0.50-0.80"
    else:
        action = allow_transaction(txn_id, amount, final_prob, source)
        reason = f"prob {final_prob:.4f} < 0.50"

    log_to_ledger(txn_id, amount, hour, source, ulb_prob, ieee_prob, final_prob, action, reason)

    return {'transaction_id':txn_id,'source':source,'amount':amount,
            'ulb_prob':round(ulb_prob,4),'ieee_prob':round(ieee_prob,4),
            'final_prob':round(final_prob,4),'action':action,'reason':reason}

In [ ]:
# Load test data from both datasets
X_test_ulb = np.array(joblib.load('../data/X_test.pkl'))
y_test_ulb = np.array(joblib.load('../data/y_test.pkl'))

print("Loading IEEE test sample...")
txn     = pd.read_csv('../data/ieee/train_transaction.csv', nrows=5000)
idn     = pd.read_csv('../data/ieee/train_identity.csv')
df_ieee = txn.merge(idn, on='TransactionID', how='left')
df_num  = df_ieee.select_dtypes(include=[np.number]).copy()
y_ieee  = df_num['isFraud'].copy()
X_ieee  = df_num.drop(columns=['isFraud','TransactionID'], errors='ignore')
null_pct = X_ieee.isnull().mean()
X_ieee  = X_ieee.drop(columns=null_pct[null_pct>0.5].index)
X_ieee  = X_ieee.fillna(X_ieee.median()).fillna(0)
ieee_feats = xgb_ieee.get_booster().feature_names
X_ieee  = X_ieee.reindex(columns=ieee_feats, fill_value=0)
X_test_ieee = np.array(X_ieee)
y_test_ieee = np.array(y_ieee)

ulb_fraud  = np.where(y_test_ulb==1)[0]
ulb_normal = np.where(y_test_ulb==0)[0]
ieee_fraud  = np.where(y_test_ieee==1)[0]
ieee_normal = np.where(y_test_ieee==0)[0]

ulb_n  = X_test_ulb.shape[1]
ieee_n = X_test_ieee.shape[1]
print(f"ULB features: {ulb_n} | IEEE features: {ieee_n}")
print(f"ULB fraud cases: {len(ulb_fraud)} | IEEE fraud cases: {len(ieee_fraud)}")

In [ ]:
# 8 test scenarios — 4 ULB + 4 IEEE
scenarios = [
    (X_test_ulb[ulb_fraud[0]],  pad_features(X_test_ulb[ulb_fraud[0]],  ieee_n), {'id':'ULB_001','amount':18500,'hour':2},  'ULB-CreditCard'),
    (X_test_ulb[ulb_fraud[1]],  pad_features(X_test_ulb[ulb_fraud[1]],  ieee_n), {'id':'ULB_002','amount':9200, 'hour':3},  'ULB-CreditCard'),
    (X_test_ulb[ulb_normal[0]], pad_features(X_test_ulb[ulb_normal[0]], ieee_n), {'id':'ULB_003','amount':250,  'hour':14}, 'ULB-CreditCard'),
    (X_test_ulb[ulb_normal[1]], pad_features(X_test_ulb[ulb_normal[1]], ieee_n), {'id':'ULB_004','amount':1100, 'hour':11}, 'ULB-CreditCard'),
    (pad_features(X_test_ieee[ieee_fraud[0]],  ulb_n), X_test_ieee[ieee_fraud[0]],  {'id':'IEEE_005','amount':24900,'hour':1},  'IEEE-CIS'),
    (pad_features(X_test_ieee[ieee_fraud[1]],  ulb_n), X_test_ieee[ieee_fraud[1]],  {'id':'IEEE_006','amount':7600, 'hour':23}, 'IEEE-CIS'),
    (pad_features(X_test_ieee[ieee_normal[0]], ulb_n), X_test_ieee[ieee_normal[0]], {'id':'IEEE_007','amount':420,  'hour':10}, 'IEEE-CIS'),
    (pad_features(X_test_ieee[ieee_normal[1]], ulb_n), X_test_ieee[ieee_normal[1]], {'id':'IEEE_008','amount':875,  'hour':16}, 'IEEE-CIS'),
]

print("="*55)
print("  DUAL-MODEL AGENTIC AI — LIVE SIMULATION")
print("="*55)

all_results = []
for f_ulb, f_ieee, meta, source in scenarios:
    result = agentic_decision_engine(f_ulb, f_ieee, meta, source)
    all_results.append(result)

print("\n✅ Simulation complete!")

In [ ]:
# Decision summary
results_df = pd.DataFrame(all_results)
print("\n=== DECISION SUMMARY ===")
print(results_df[['transaction_id','source','amount','ulb_prob','ieee_prob','final_prob','action']].to_string(index=False))
print("\n=== ACTIONS BY DATASET ===")
print(results_df.groupby(['source','action']).size().reset_index(name='count').to_string(index=False))

In [ ]:
# Visualization
color_map = {'BLOCK':'crimson','OTP_CHALLENGE':'orange','ALLOW':'seagreen'}
bar_colors = [color_map[a] for a in results_df['action']]
fig, axes = plt.subplots(1, 3, figsize=(16,5))

x = np.arange(len(results_df)); w = 0.28
axes[0].bar(x-w, results_df['ulb_prob'],   w, label='ULB',   color='steelblue', alpha=0.8)
axes[0].bar(x,   results_df['ieee_prob'],  w, label='IEEE',  color='darkorange',alpha=0.8)
axes[0].bar(x+w, results_df['final_prob'], w, label='Final', color='purple',    alpha=0.85)
axes[0].axhline(y=0.8, color='crimson', linestyle='--', lw=1.5, label='Block(0.8)')
axes[0].axhline(y=0.5, color='orange',  linestyle='--', lw=1.5, label='OTP(0.5)')
axes[0].set_title('ULB vs IEEE vs Final Probability')
axes[0].set_ylim(0,1.1); axes[0].legend(fontsize=7)
axes[0].set_xticks(x); axes[0].set_xticklabels(results_df['transaction_id'], rotation=35, ha='right', fontsize=7)

action_counts = results_df['action'].value_counts()
pie_colors = [color_map.get(a,'gray') for a in action_counts.index]
axes[1].pie(action_counts.values, labels=action_counts.index, colors=pie_colors, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Action Distribution')

axes[2].scatter(results_df['ulb_prob'], results_df['ieee_prob'], c=bar_colors, s=140, edgecolors='black', zorder=5)
axes[2].plot([0,1],[0,1],'k--',alpha=0.4)
axes[2].set_xlabel('ULB Probability'); axes[2].set_ylabel('IEEE Probability')
axes[2].set_title('Model Agreement'); axes[2].set_xlim(0,1); axes[2].set_ylim(0,1)
patches = [mpatches.Patch(color=v,label=k) for k,v in color_map.items()]
axes[2].legend(handles=patches, fontsize=7)

plt.tight_layout()
plt.savefig('../reports/phase5_agentic_decisions.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
# View ledger and save system
ledger = pd.read_csv(LEDGER_PATH)
print(f"=== FRAUD LEDGER — {len(ledger)} decisions ===")
print(ledger.to_string(index=False))
ledger.to_csv('../reports/fraud_ledger_snapshot.csv', index=False)

agent_package = {
    'ulb_model'  : xgb_ulb,
    'ieee_model' : xgb_ieee,
    'scaler'     : scaler,
    'thresholds' : {'block':0.8,'otp':0.5},
    'ulb_weight' : 0.5, 'ieee_weight': 0.5,
    'version'    : '2.0',
    'created'    : datetime.now().strftime('%Y-%m-%d')
}
joblib.dump(agent_package, '../models/agentic_system.pkl')
print("\nAgentic system saved to models/agentic_system.pkl")
print("Phase 5 complete!")